# Baseline eval of the unpruned teacher: MBPP-TS

Part of plan.md step 1.2's baseline eval suite (`Qwen3-30B-A3B-Base` as-is --
the number every pruned candidate in Phase 2/4 gets compared to), covering
the other **primary, gates-every-decision** task: MBPP-TS (`multiple-mbpp-ts`,
registered by `bonsai.eval.register` -- `bigcode-evaluation-harness` only
wraps the HumanEval translations natively), pass@1 at temperature 0.2 /
nucleus top-p 0.95. Run independently from HumanEval-TS and the
general-knowledge suite (each gets its own notebook) so a failure or a long
run in one doesn't block or hide the others.

Same config as `02a_baseline_humaneval_ts.ipynb`: `batch_size=8`,
`max_length_generation=512` -- at `batch_size=32` batched generation waits
for the slowest sequence in the batch to stop, and this MoE model's decode
cost doesn't amortize across a batch as cleanly as a dense model's would,
which stalled HumanEval-TS for over an hour before this fix.

MBPP-TS has 390 problems vs. HumanEval-TS's 159, so expect roughly 2.5x the
wall time (~1 hour, going by HumanEval-TS's ~24 minutes).

Needs a rented GPU (1x 80GB, per plan.md's sizing table) with
`WITH_EVAL=1 ./tools/setup_instance.sh` (installs the `eval` extra plus
Node.js + `typescript` -- MultiPL-E's TS execution shells out to `tsc` and
`node`, neither of which the plain Python env ships).

In [1]:
import json

from bonsai.config import load_config
from bonsai.tracking import new_run, save_config_snapshot

config = load_config("../configs/qwen3-30b-a3b.toml")
run_dir = new_run("baseline-eval")
save_config_snapshot(run_dir, config)

print(f"model: {config.model.repo_id}")
print(f"run dir: {run_dir}")

model: Qwen/Qwen3-30B-A3B-Base
run dir: runs/20260918T171357Z-8b1038-baseline-eval


In [2]:
import os

os.environ["MODEL_REPO_ID"] = config.model.repo_id
os.environ["MODEL_REVISION"] = config.model.revision
os.environ["RUN_DIR"] = str(run_dir)

In [3]:
!stdbuf -o0 -e0 accelerate launch ../tools/bigcode_eval_main.py \
    --model "$MODEL_REPO_ID" --revision "$MODEL_REVISION" --precision bf16 \
    --tasks multiple-mbpp-ts \
    --max_length_generation 512 --batch_size 8 \
    --allow_code_execution --save_generations \
    --metric_output_path "$RUN_DIR/mbpp-ts.json" \
    --save_generations_path "$RUN_DIR/mbpp-ts-generations.json" \
    2>&1 | tee /tmp/live_mbpp_ts.log

assert _exit_code == 0, "accelerate launch failed for multiple-mbpp-ts"  # noqa: F821 -- set by the ! shell magic above at runtime

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.


Selected Tasks: ['multiple-mbpp-ts']
Loading model in bf16
Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

Loading weights:   1%|          | 4/531 [00:00<00:33, 15.62it/s]

Loading weights:   1%|          | 6/531 [00:01<01:57,  4.48it/s]

Loading weights:   3%|▎         | 15/531 [00:01<00:41, 12.51it/s]

Loading weights:   3%|▎         | 18/531 [00:02<01:04,  7.96it/s]

... (872 outputs left out)

 99%|█████████▉| 388/390 [03:52<00:02,  1.15s/it]

100%|█████████▉| 389/390 [03:52<00:00,  1.08it/s]

100%|██████████| 390/390 [03:53<00:00,  1.67it/s]


{
  "multiple-mbpp-ts": {
    "pass@1": 0.7333333333333333
  },
  "config": {
    "prefix": "",
    "do_sample": true,
    "temperature": 0.2,
    "top_k": 0,
    "top_p": 0.95,
    "n_samples": 1,
    "eos": "<|endoftext|>",
    "seed": 0,
    "model": "Qwen/Qwen3-30B-A3B-Base",
    "modeltype": "causal",
    "peft_model": null,
    "revision": "main",
    "use_auth_token": false,
    "trust_remote_code": false,
    "tasks": "multiple-mbpp-ts",
    "instruction_tokens": null,
    "batch_size": 8,
    "max_length_generation": 512,
    "precision": "bf16",
    "load_in_8bit": false,
    "load_in_4bit": false,
    "left_padding": false,
    "limit": null,
    "limit_start": 0,
    "save_every_k_tasks": -1,
    "postprocess": true,
    "allow_code_execution": true,
    "generation_only": false,
    "load_generations_path": null,
    "load_data_path": null,
    "metric_output_path": "runs/20260918T171357Z-8b1038-baseline-eval/mbpp-ts.json",
    "save_generations": true,
    "load_generatio

In [4]:
mbpp_ts = json.loads((run_dir / "mbpp-ts.json").read_text())
print("mbpp-ts:", mbpp_ts)

mbpp-ts: {'multiple-mbpp-ts': {'pass@1': 0.7333333333333333}, 'config': {'prefix': '', 'do_sample': True, 'temperature': 0.2, 'top_k': 0, 'top_p': 0.95, 'n_samples': 1, 'eos': '<|endoftext|>', 'seed': 0, 'model': 'Qwen/Qwen3-30B-A3B-Base', 'modeltype': 'causal', 'peft_model': None, 'revision': 'main', 'use_auth_token': False, 'trust_remote_code': False, 'tasks': 'multiple-mbpp-ts', 'instruction_tokens': None, 'batch_size': 8, 'max_length_generation': 512, 'precision': 'bf16', 'load_in_8bit': False, 'load_in_4bit': False, 'left_padding': False, 'limit': None, 'limit_start': 0, 'save_every_k_tasks': -1, 'postprocess': True, 'allow_code_execution': True, 'generation_only': False, 'load_generations_path': None, 'load_data_path': None, 'metric_output_path': 'runs/20260918T171357Z-8b1038-baseline-eval/mbpp-ts.json', 'save_generations': True, 'load_generations_intermediate_paths': None, 'save_generations_path': 'runs/20260918T171357Z-8b1038-baseline-eval/mbpp-ts-generations.json', 'save_refer